<a href="https://colab.research.google.com/github/MeenakshiRajpurohit/CMPE-255-Data-Mining/blob/main/hw4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANALYTICAL ASSIGNMENT
**(55pt)**

You need to write answers for all the questions clearly, with diagrams, schemas, and SQL queries where required.

Submit this part as PDF file following instructions at the end of this document.



## ASSOCIATION PATTERN MINING

### Problem 1 (20pt)

A database has 5 transactions. Let **min_sup = 60%** and **min_conf = 80%**.

| TID  | Items Bought          |
|------|-----------------------|
| T100 | {M, O, N, K, E, Y}    |
| T200 | {D, O, N, K, E, Y}    |
| T300 | {M, A, K, E}          |
| T400 | {M, U, C, K, Y}       |
| T500 | {C, O, O, K, I, E}    |

1. Find all frequent itemsets using **Apriori** method.
2. Find all frequent itemsets using **FP-growth**.
3. Compare the efficiency of the two mining processes.
4. List all of the strong association rules (with support *s* and confidence *c*) matching the following metarule, where *X* is a variable representing customers, and *item<sub>i</sub>* denotes variables representing items (e.g., "A", "B", etc.):

$$\forall\, x \in \text{transaction},\ \text{buys}(X, item_1) \land \text{buys}(X, item_2) \Rightarrow \text{buys}(X, item_3)\ [s, c]$$

### Association Pattern Mining Solution



**Database:**

| TID | Items Bought |
|-----|--------------|
| T100 | {M, O, N, K, E, Y} |
| T200 | {D, O, N, K, E, Y} |
| T300 | {M, A, K, E} |
| T400 | {M, U, C, K, Y} |
| T500 | {C, O, O, K, I, E} |


- Total transactions = 5
- min_sup = 60% → minimum support count = 0.60 × 5 = **3**
- min_conf = 80%

Note: In T500, "O" appears twice but in itemset mining we treat items as a set, so {C, O, K, I, E} is used for counting.



### Part (a): Apriori Method

#### Step 1: Generate C₁ and find L₁ (1-itemsets)

Scan database to count each item:

| Item | Count | Transactions |
|------|-------|--------------|
| M | 3 | T100, T300, T400 |
| O | 3 | T100, T200, T500 |
| N | 2 | T100, T200 |
| K | 5 | T100, T200, T300, T400, T500 |
| E | 4 | T100, T200, T300, T500 |
| Y | 3 | T100, T200, T400 |
| D | 1 | T200 |
| A | 1 | T300 |
| U | 1 | T400 |
| C | 2 | T400, T500 |
| I | 1 | T500 |

**L₁ (items with count ≥ 3):**

| Itemset | Support Count |
|---------|---------------|
| {M} | 3 |
| {O} | 3 |
| {K} | 5 |
| {E} | 4 |
| {Y} | 3 |

### Step 2: Generate C₂ and find L₂ (2-itemsets)

Candidates from L₁ × L₁:

| Itemset | Count | Transactions |
|---------|-------|--------------|
| {M, O} | 1 | T100 |
| {M, K} | 3 | T100, T300, T400 |
| {M, E} | 2 | T100, T300 |
| {M, Y} | 2 | T100, T400 |
| {O, K} | 3 | T100, T200, T500 |
| {O, E} | 3 | T100, T200, T500 |
| {O, Y} | 2 | T100, T200 |
| {K, E} | 4 | T100, T200, T300, T500 |
| {K, Y} | 3 | T100, T200, T400 |
| {E, Y} | 2 | T100, T200 |

**L₂ (count ≥ 3):**

| Itemset | Support Count |
|---------|---------------|
| {M, K} | 3 |
| {O, K} | 3 |
| {O, E} | 3 |
| {K, E} | 4 |
| {K, Y} | 3 |

### Step 3: Generate C₃ and find L₃ (3-itemsets)

Join L₂ ⋈ L₂ and apply pruning (all 2-subsets must be in L₂):

- {M, K, E} → subset {M, E} ∉ L₂ → **pruned**
- {M, K, Y} → subset {M, Y} ∉ L₂ → **pruned**
- {O, K, E} → subsets {O,K}, {O,E}, {K,E} all in L₂ → **keep**
- {O, K, Y} → subset {O, Y} ∉ L₂ → **pruned**
- {K, E, Y} → subset {E, Y} ∉ L₂ → **pruned**

Count {O, K, E}: present in T100, T200, T500 → count = 3 ✓

**L₃:**

| Itemset | Support Count |
|---------|---------------|
| {O, K, E} | 3 |

### Step 4: Generate C₄

L₃ has only one itemset, so no 4-itemsets can be formed. **Algorithm terminates.**

### Final Frequent Itemsets (Apriori)

$$L = L_1 \cup L_2 \cup L_3$$

**1-itemsets:** {M}:3, {O}:3, {K}:5, {E}:4, {Y}:3  
**2-itemsets:** {M,K}:3, {O,K}:3, {O,E}:3, {K,E}:4, {K,Y}:3  
**3-itemsets:** {O,K,E}:3


## Part (b): FP-Growth Method

### Step 1: Build the ordered frequent item list

Sort L₁ by descending support count (ties broken alphabetically):

**F-list (header order): K:5, E:4, M:3, O:3, Y:3**

### Step 2: Reorder transactions (keeping only frequent items, in F-list order)

| TID | Original | Filtered & Ordered |
|-----|----------|--------------------|
| T100 | {M, O, N, K, E, Y} | {K, E, M, O, Y} |
| T200 | {D, O, N, K, E, Y} | {K, E, O, Y} |
| T300 | {M, A, K, E} | {K, E, M} |
| T400 | {M, U, C, K, Y} | {K, M, Y} |
| T500 | {C, O, O, K, I, E} | {K, E, O} |

### Step 3: Build the FP-Tree

                null
                  
                 │
                 K:5
                 │
          ┌──────┴──────┐
          E:4           M:1
          │             │
    ┌─────┼─────┐       Y:1
    M:1   O:2   M:1
    │     │
    O:1   Y:1
    │
    Y:1

**Tree construction trace:**
- T100 {K,E,M,O,Y}: K:1 → E:1 → M:1 → O:1 → Y:1
- T200 {K,E,O,Y}: K:2 → E:2 → O:1 → Y:1 (new branch under E)
- T300 {K,E,M}: K:3 → E:3 → M:2 (M increments under E)
- T400 {K,M,Y}: K:4 → M:1 → Y:1 (new branch under K)
- T500 {K,E,O}: K:5 → E:4 → O:2 (O increments under E)

### Step 4: Header Table with Node Links

| Item | Count | Node-link path |
|------|-------|----------------|
| K | 5 | K:5 |
| E | 4 | E:4 |
| M | 3 | M:1 (under E:M), M:2 (under E:M after T300), M:1 (under K direct) |
| O | 3 | O:2 (under E), O:1 (under E:M) |
| Y | 3 | Y:1 (under E:O), Y:1 (under E:M:O), Y:1 (under K:M) |

### Step 5: Mine the FP-Tree (bottom-up from F-list)

#### Mining suffix **Y** (count = 3)
**Conditional Pattern Base for Y:**
- {K, E, M, O}: 1
- {K, E, O}: 1
- {K, M}: 1

Count items: K:3, E:2, M:2, O:2

Only K meets min_sup = 3. **Conditional FP-tree for Y:** ⟨K:3⟩

**Frequent patterns:** {Y}:3, **{K, Y}:3**

#### Mining suffix **O** (count = 3)
**Conditional Pattern Base for O:**
- {K, E}: 2
- {K, E, M}: 1

Count items: K:3, E:3, M:1

**Conditional FP-tree for O:** ⟨K:3 → E:3⟩

**Frequent patterns:** {O}:3, **{K,O}:3, {E,O}:3, {K,E,O}:3**

#### Mining suffix **M** (count = 3)
**Conditional Pattern Base for M:**
- {K, E}: 2
- {K}: 1

Count items: K:3, E:2

**Conditional FP-tree for M:** ⟨K:3⟩

**Frequent patterns:** {M}:3, **{K, M}:3**

#### Mining suffix **E** (count = 4)
**Conditional Pattern Base for E:**
- {K}: 4

**Conditional FP-tree for E:** ⟨K:4⟩

**Frequent patterns:** {E}:4, **{K, E}:4**

#### Mining suffix **K** (count = 5)
No prefix path. **Frequent pattern:** {K}:5

### Final Frequent Itemsets (FP-Growth)

| Itemset | Support |
|---------|---------|
| {K} | 5 |
| {E} | 4 |
| {M}, {O}, {Y} | 3 each |
| {K, E} | 4 |
| {K, M} | 3 |
| {K, O}, {E, O} | 3 each |
| {K, Y} | 3 |
| {K, E, O} | 3 |

**Result is identical to Apriori**



## Part (c): Efficiency Comparison

| Aspect | Apriori | FP-Growth |
|--------|---------|-----------|
| **Database scans** | Multiple (k+1 scans for L_k) — here **4 scans** | Only **2 scans** (one for F-list, one to build tree) |
| **Candidate generation** | Generates many candidates (\|C₂\| = 10, \|C₃\| = 5, with pruning) | **No candidate generation** — patterns mined directly from tree |
| **Memory** | Lower per pass; only needs current candidates | Higher upfront — stores compact FP-tree in memory |
| **Computation** | Repeated subset checking against transactions; expensive at level 2 | Recursive tree traversal with conditional pattern bases |
| **Suitability for this dataset** | Works fine for 5 transactions | Slight overhead for tiny data; advantage grows with database size |

**Conclusion:** For this small database both methods reach the same answer with comparable wall-clock effort, but FP-growth scales far better — its 2-scan, no-candidate-generation design makes it dramatically faster on large, dense datasets where Apriori's level-wise candidate explosion (especially at C₂) becomes the bottleneck. Apriori's main weakness here was generating 10 candidates at level 2 to confirm only 5 frequent pairs.



## Part (d): Strong Association Rules

The metarule is `buys(X, item1) ∧ buys(X, item2) ⇒ buys(X, item3)` — a rule with **2 items in the antecedent and 1 item in the consequent**, derived from a **3-itemset**.

The only frequent 3-itemset is **{O, K, E}** with support count 3 (support = 60%).

For each 2-item subset as antecedent, compute confidence = sup(X∪Y)/sup(X):

| Rule | Support | Confidence | Calculation | Strong? |
|------|---------|------------|-------------|---------|
| O ∧ K ⇒ E | 3/5 = 60% | 3/3 = **100%** | sup{O,K,E}/sup{O,K} = 3/3 | Yes |
| O ∧ E ⇒ K | 3/5 = 60% | 3/3 = **100%** | sup{O,K,E}/sup{O,E} = 3/3 | Yes |
| K ∧ E ⇒ O | 3/5 = 60% | 3/4 = **75%** | sup{O,K,E}/sup{K,E} = 3/4 | No (< 80%) |

### Strong Rules Matching the Metarule

$$\boxed{\text{buys}(X, O) \land \text{buys}(X, K) \Rightarrow \text{buys}(X, E) \quad [s = 60\%, \; c = 100\%]}$$

$$\boxed{\text{buys}(X, O) \land \text{buys}(X, E) \Rightarrow \text{buys}(X, K) \quad [s = 60\%, \; c = 100\%]}$$

The third candidate `K ∧ E ⇒ O` has confidence 75%, which falls below the 80% threshold, so it is **not** a strong rule.


## Problem 2 (15pt)

A database has four transactions. Let min_sup = 60% and min_conf = 80%.

| Cust ID | TID  | Items Bought (in the form of brand-item category) |
|---------|------|----------------------------------------------------|
| 01      | T100 | {King's-Crab, Sunset-Milk, Dairyland-Cheese, Best-Bread} |
| 02      | T200 | {Best-Cheese, Dairyland-Milk, Goldenfarm-Apple, Tasty-Pie, Wonder-Bread} |
| 03      | T300 | {Westcoast-Apple, Dairyland-Milk, Wonder-Bread, Tasty-Pie} |
| 04      | T400 | {Wonder-Bread, Sunset-Milk, Dairyland-Cheese} |

- Total transactions = 4
- min_sup = 60% → minimum support count = ⌈0.60 × 4⌉ = 3
- min_conf = 80%

### Part (1): Frequent k-itemset for the largest k — at item category granularity

| TID  | Item Categories                       |
|------|---------------------------------------|
| T100 | {Crab, Milk, Cheese, Bread}           |
| T200 | {Cheese, Milk, Apple, Pie, Bread}     |
| T300 | {Apple, Milk, Bread, Pie}             |
| T400 | {Bread, Milk, Cheese}                 |

#### Step 1: Count 1-itemsets

| Item   | Count | Transactions             | Frequent? (≥ 3) |
|--------|-------|--------------------------|-----------------|
| Crab   | 1     | T100                     | No  |
| Milk   | 4     | T100, T200, T300, T400   | Yes |
| Cheese | 3     | T100, T200, T400         | Yes |
| Bread  | 4     | T100, T200, T300, T400   | Yes |
| Apple  | 2     | T200, T300               | No  |
| Pie    | 2     | T200, T300               | No  |

**L₁ = { {Milk}:4, {Cheese}:3, {Bread}:4 }**

#### Step 2: Count 2-itemsets

| Itemset           | Count | Transactions             | Frequent? |
|-------------------|-------|--------------------------|-----------|
| {Milk, Cheese}    | 3     | T100, T200, T400         | Yes |
| {Milk, Bread}     | 4     | T100, T200, T300, T400   | Yes |
| {Cheese, Bread}   | 3     | T100, T200, T400         | Yes |

**L₂ = { {Milk, Cheese}:3, {Milk, Bread}:4, {Cheese, Bread}:3 }**

#### Step 3: Count 3-itemsets

| Itemset                  | Count | Transactions       | Frequent? |
|--------------------------|-------|--------------------|-----------|
| {Milk, Cheese, Bread}    | 3     | T100, T200, T400   | Yes |

**L₃ = { {Milk, Cheese, Bread}:3 }**

#### Step 4: 4-itemsets

No more frequent items available beyond these three categories → algorithm terminates.

#### Frequent k-itemset for the largest k

$$\boxed{\{\text{Milk, Cheese, Bread}\} \text{ with support count } 3 \;(s = 75\%)}$$

The largest k is **k = 3**.



### Part (2): Strong association rules containing the largest frequent k-itemset

For the 3-itemset {Milk, Cheese, Bread} (sup = 3/4 = 75%), generate all rules of the form *Antecedent ⇒ Consequent*. Confidence = sup(A ∪ C) / sup(A).

**Required supports from L₂ and L₁:**
- sup{Milk, Cheese} = 3
- sup{Milk, Bread} = 4
- sup{Cheese, Bread} = 3
- sup{Milk} = 4
- sup{Cheese} = 3
- sup{Bread} = 4

| Rule                                  | Support | Confidence       | Strong? (≥ 80%) |
|---------------------------------------|---------|------------------|-----------------|
| Milk ∧ Cheese ⇒ Bread                 | 75%     | 3/3 = **100%**   | Yes |
| Milk ∧ Bread ⇒ Cheese                 | 75%     | 3/4 = **75%**    | No  |
| Cheese ∧ Bread ⇒ Milk                 | 75%     | 3/3 = **100%**   | Yes |
| Milk ⇒ Cheese ∧ Bread                 | 75%     | 3/4 = **75%**    | No  |
| Cheese ⇒ Milk ∧ Bread                 | 75%     | 3/3 = **100%**   | Yes |
| Bread ⇒ Milk ∧ Cheese                 | 75%     | 3/4 = **75%**    | No  |

#### Strong Association Rules

$$\boxed{\text{buys}(X, \text{Milk}) \land \text{buys}(X, \text{Cheese}) \Rightarrow \text{buys}(X, \text{Bread}) \quad [s = 75\%,\; c = 100\%]}$$

$$\boxed{\text{buys}(X, \text{Cheese}) \land \text{buys}(X, \text{Bread}) \Rightarrow \text{buys}(X, \text{Milk}) \quad [s = 75\%,\; c = 100\%]}$$

$$\boxed{\text{buys}(X, \text{Cheese}) \Rightarrow \text{buys}(X, \text{Milk}) \land \text{buys}(X, \text{Bread}) \quad [s = 75\%,\; c = 100\%]}$$


### Part (3): Frequent k-itemset for the largest k — at brand-item category granularity

Now use the original brand-item names. The **rule template references customers**, so each customer is the unit of analysis. Since there are 4 customers (each with one transaction), the support count is the same as before: minimum count = 3.

Count each brand-item across customers:

| Brand-Item        | Count | Customers       | Frequent? (≥ 3) |
|-------------------|-------|------------------|-----------------|
| King's-Crab       | 1     | 01               | No  |
| Sunset-Milk       | 2     | 01, 04           | No  |
| Dairyland-Cheese  | 2     | 01, 04           | No  |
| Best-Bread        | 1     | 01               | No  |
| Best-Cheese       | 1     | 02               | No  |
| Dairyland-Milk    | 2     | 02, 03           | No  |
| Goldenfarm-Apple  | 1     | 02               | No  |
| Tasty-Pie         | 2     | 02, 03           | No  |
| Wonder-Bread      | 3     | 02, 03, 04       | Yes |
| Westcoast-Apple   | 1     | 03               | No  |

Only **{Wonder-Bread}** meets the minimum support of 3.

Since only one item is frequent, no 2-itemsets (or higher) can satisfy minimum support.

#### Frequent k-itemset for the largest k

$$\boxed{\{\text{Wonder-Bread}\} \text{ with support count } 3 \;(s = 75\%)}$$

The largest k is **k = 1**.

> The brand-item granularity is far sparser than the category-level view: brand fragmentation (e.g., Sunset-Milk vs. Dairyland-Milk) splits the support of what would otherwise be a frequent category like "Milk." This illustrates the **multilevel association mining** trade-off — mining at higher abstraction levels reveals patterns that disappear under fine-grained brand labels.
````

## Problem 3 (20pt)

### Contingency Table

|              | hot dogs | ¬hot dogs | Σ row |
|--------------|----------|-----------|-------|
| hamburgers   | 2000     | 500       | 2500  |
| ¬hamburgers  | 1000     | 1500      | 2500  |
| Σ col        | 3000     | 2000      | 5000  |

Total transactions: **N = 5000**



### Part (a): Is the rule "hot dogs ⇒ hamburgers" strong?

A rule is **strong** if it satisfies both *min_sup* and *min_conf*.

#### Support

$$
\text{support}(\text{hot dogs} \Rightarrow \text{hamburgers}) = \frac{P(\text{hot dogs} \cap \text{hamburgers})}{N} = \frac{2000}{5000} = 0.40 = 40\%
$$

Since 40% ≥ 25% → **support threshold satisfied**.

#### Confidence

$$
\text{confidence}(\text{hot dogs} \Rightarrow \text{hamburgers}) = \frac{\text{sup}(\text{hot dogs} \cap \text{hamburgers})}{\text{sup}(\text{hot dogs})} = \frac{2000}{3000} \approx 0.667 = 66.7\%
$$

Since 66.7% ≥ 50% → **confidence threshold satisfied**.

#### Conclusion

$$\boxed{\text{Yes, the rule is strong: } s = 40\%,\; c \approx 66.7\%}$$



### Part (b): Are hot dogs and hamburgers independent? If not, what kind of correlation?

Two events A and B are **independent** if and only if  
$P(A \cap B) = P(A) \cdot P(B)$, equivalently $\text{lift}(A, B) = 1$.

#### Compute the marginal probabilities

- $P(\text{hot dogs}) = 3000/5000 = 0.60$
- $P(\text{hamburgers}) = 2500/5000 = 0.50$
- $P(\text{hot dogs} \cap \text{hamburgers}) = 2000/5000 = 0.40$

#### Independence check

$$
P(\text{hot dogs}) \cdot P(\text{hamburgers}) = 0.60 \times 0.50 = 0.30
$$

But $P(\text{hot dogs} \cap \text{hamburgers}) = 0.40 \ne 0.30$.

→ **They are NOT independent.**

#### Lift (correlation measure)

$$
\text{lift}(\text{hot dogs}, \text{hamburgers}) = \frac{P(A \cap B)}{P(A) \cdot P(B)} = \frac{0.40}{0.30} \approx 1.33
$$

Since lift > 1, the two items are **positively correlated** buying hot dogs makes a customer *more likely* to buy hamburgers than chance would predict.

$$\boxed{\text{Lift} \approx 1.33 > 1 \;\Rightarrow\; \text{positively correlated}}$$



### Part (c): Comparison with all_confidence, max_confidence, Kulczynski, and cosine

Let A = hot dogs, B = hamburgers, with:
- sup(A) = 3000, sup(B) = 2500, sup(A ∩ B) = 2000

#### Formulas

| Measure          | Formula                                                                |
|------------------|------------------------------------------------------------------------|
| all_confidence   | $\dfrac{\text{sup}(A \cap B)}{\max(\text{sup}(A),\text{sup}(B))}$       |
| max_confidence   | $\max\left(\dfrac{\text{sup}(A \cap B)}{\text{sup}(A)},\;\dfrac{\text{sup}(A \cap B)}{\text{sup}(B)}\right)$ |
| Kulczynski       | $\dfrac{1}{2}\left(\dfrac{\text{sup}(A \cap B)}{\text{sup}(A)}+\dfrac{\text{sup}(A \cap B)}{\text{sup}(B)}\right)$ |
| cosine           | $\dfrac{\text{sup}(A \cap B)}{\sqrt{\text{sup}(A)\cdot\text{sup}(B)}}$  |

#### Calculations

**all_confidence:**
$$
\text{all\_conf} = \frac{2000}{\max(3000, 2500)} = \frac{2000}{3000} \approx 0.667
$$

**max_confidence:**
$$
\text{max\_conf} = \max\!\left(\frac{2000}{3000},\;\frac{2000}{2500}\right) = \max(0.667,\; 0.800) = 0.800
$$

**Kulczynski:**
$$
\text{Kulc} = \frac{1}{2}\left(\frac{2000}{3000} + \frac{2000}{2500}\right) = \frac{1}{2}(0.667 + 0.800) \approx 0.733
$$

**cosine:**
$$
\text{cosine} = \frac{2000}{\sqrt{3000 \times 2500}} = \frac{2000}{\sqrt{7\,500\,000}} \approx \frac{2000}{2738.6} \approx 0.730
$$

#### Summary table

| Measure          | Value     | Range     | Interpretation |
|------------------|-----------|-----------|----------------|
| Lift             | **1.33**  | [0, ∞)    | > 1 → positive correlation (depends on N) |
| Correlation (χ² / φ) | positive  | —         | A and B are positively correlated |
| all_confidence   | **0.667** | [0, 1]    | Moderately strong association |
| max_confidence   | **0.800** | [0, 1]    | Strong from B's side ($P(A\|B)=0.8$) |
| Kulczynski       | **0.733** | [0, 1]    | Balanced average — strong |
| cosine           | **0.730** | [0, 1]    | Strong similarity |

#### Discussion

- **Lift** correctly reports a *positive* correlation, but its value depends on the **total number of transactions N** (the count of "null transactions" — those containing neither A nor B). This makes it sensitive to the database size and not **null-invariant**.
- **all_confidence, max_confidence, Kulczynski, and cosine** are all **null-invariant** measures: they ignore the count of transactions containing neither item (in this case, 1500). Their values do not change if we add or remove transactions that contain neither hot dogs nor hamburgers.
- All four null-invariant measures here fall in the range **0.67 – 0.80**, agreeing with lift that the association is genuinely strong and positive.
- **Kulczynski (≈ 0.73)** is widely recommended for skewed or imbalanced data because it averages the two conditional probabilities and gives a balanced view.
- **max_confidence (0.80)** is the most optimistic — it reports the stronger of the two directional confidences and can be misleading when one direction is weak.
- **all_confidence (0.67)** is the most conservative — bounded by the weaker direction.
- **cosine (0.73)** behaves like a geometric mean of the two confidences and tracks Kulczynski closely here.


$$\boxed{\text{All measures confirm a positive, strong association between hot dogs and hamburgers.}}$$

The null-invariant measures (all_conf, max_conf, Kulczynski, cosine) are generally preferred over lift and χ²-based correlation in transactional data, because they are not distorted by the typically huge number of transactions containing neither item — a property especially important in sparse, large-scale market-basket analysis.